# Convolutional Neural Networks: Application (2)

In this notebook, you will:
- Build a ConvNet to **Identify Sign Language Digits (Multiclass)** using the TF Keras Functional API

**After this assignment we will be able to:**
- Build and train a ConvNet in TensorFlow for a __multiclass__ classification problem

# Load Library

In [1]:
import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd
import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.python.framework import ops

# Identify Sign Language Digits (Multiclass)

We'll use Keras' flexible [Functional API](https://www.tensorflow.org/guide/keras/functional) to build a **ConvNet** that can **differentiate** between **6 sign language digits**.

<img src="images/SIGNS.png" style="width:800px;height:300px;">
<caption><center> <u> <font color='purple'> <b>Figure 1</b> </u><font color='purple'>  : <b>Identify Sign Language Digits</b><br> with one-hot represent for each class </center></caption>

In the visual example below, the one possible direction of the movement Sequential model is shown in contrast to a skip connection, which is just one of the many ways a Functional model can be constructed.

<font color="purple">

**Skip connection** = **skips** some **layer** in the network and **feeds** the **output** to a **later layer** in the network.

</font>

<img src="images/seq_vs_func.png" style="width:350px;height:200px;">

## 1. Load Data and Split into Train/Test Sets

As a reminder, the **SIGNS dataset** is a **collection of 6 signs** representing numbers from **0 to 5**.

### 1.1 - Load Data from .h5 File

In [2]:
# ======= DATASETS STRUCTURE =======
train_dataset = h5py.File("datasets/train_signs.h5", "r")
test_dataset = h5py.File("datasets/test_signs.h5", "r")

def print_structure_clean(dataset, title):
    print(f"\n{title}")
    for key in dataset.keys():
        print(f"  • {key}")

print_structure_clean(train_dataset, "Train Set Structure")
print_structure_clean(test_dataset, "Test Set Structure")


Train Set Structure
  • list_classes
  • train_set_x
  • train_set_y

Test Set Structure
  • list_classes
  • test_set_x
  • test_set_y


In [3]:
# ======= TRAINING SET =======
train_set_x_orig = np.array(train_dataset["train_set_x"][:])
train_set_y_orig = np.array(train_dataset["train_set_y"][:])
print("train_set_x_orig shape:", train_set_x_orig.shape)
print("train_set_y_orig shape:", train_set_y_orig.shape)

train_set_x_orig shape: (1080, 64, 64, 3)
train_set_y_orig shape: (1080,)


In [4]:
# ======= TEST SET =======
test_set_x_orig = np.array(test_dataset["test_set_x"][:])
test_set_y_orig = np.array(test_dataset["test_set_y"][:])
print("test_set_x_orig shape:", test_set_x_orig.shape)
print("test_set_y_orig shape:", test_set_y_orig.shape)

test_set_x_orig shape: (120, 64, 64, 3)
test_set_y_orig shape: (120,)


In [5]:
# ======= CLASSES =======
list_classes = np.array(train_dataset["list_classes"][:])
print("Classes shape:", list_classes.shape)

Classes shape: (6,)


### 1.2 - Convert all 1D -> 2D Array

We need to do that because it helps:
- **Easier for compute Loss**
- **Match with Output model**

**_Note:_** We do **NOT need** to **convert classes** because it does **NOT participate** in the **calculation** nor **input** for the **model**.

In [6]:
train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))
print("train_set_y_orig shape:", train_set_y_orig.shape)
print("test_set_y_orig shape:", test_set_y_orig.shape)

train_set_y_orig shape: (1, 1080)
test_set_y_orig shape: (1, 120)


### 1.3 - Merge Label with Classes